# Baseline YOLO26n tanpa pretrained — seed 42 di Kaggle

Clone eksperimen 03 untuk membandingkan bobot acak dengan baseline pretrained pada seed yang sama.
Arsitektur tetap YOLO26n; model dibangun dari `yolo26n.yaml` dengan `PRETRAINED=False`.
Tidak ada teacher atau checkpoint awal. Bobot hasil training seed sebelumnya tidak digunakan.

Protokol baseline dipertahankan: 100 epoch per seed, imgsz 640, batch 16, AdamW lr 0.001,
AMP aktif, dataset Poltrada V3, augmentasi dan evaluasi yang sama. Validation memilih `best.pt`
dan menjadi split evaluasi akhir; test tidak digunakan untuk tuning. Ultralytics 8.4.155,
W&B 0.28.1 dan mekanisme upload per epoch mengikuti notebook baseline 03.

**Default:** `SEEDS_TO_RUN=[42]`. Seed 123, 2024, 7, dan 3407 tersedia sebagai komentar;
semuanya belum dijalankan untuk scratch. Run All membuat run W&B baru dengan tag `from-scratch`.
Setiap seed menyelesaikan training, evaluasi, upload, penutupan run dan pembersihan GPU sebelum
seed berikutnya. Setelah kegagalan, isi daftar hanya dengan seed scratch yang belum selesai.
Hasil pretrained tidak diambil, digabung, atau dianggap hasil scratch.

**AMP:** pemeriksaan bawaan Ultralytics tetap aktif dan dapat mengunduh model diagnostik pretrained.
Model pemeriksaan itu tidak mengisi bobot student. Evaluasi akhir memuat `best.pt` hasil training
scratch ini, bukan checkpoint pretrained eksternal.


## 1. Experiment Configuration

In [ ]:
# Ubah parameter di bagian ini saja untuk menjalankan eksperimen.
EXPERIMENT_ID = "03_baseline_yolo26n_scratch"
METHOD = "none"
DATA = "/kaggle/input/datasets/ilhamrafiqin/poltrada-v3-yolo26-clean/data.yaml"  # Dataset bersih di Kaggle.
MODEL_WEIGHTS = "yolo26n.yaml"  # Arsitektur nano; bukan checkpoint pretrained.
PRETRAINED = False
EPOCHS = 100
IMGSZ = 640
BATCH = 16
# Semua seed scratch merupakan eksperimen baru; mulai dengan seed 42.
SEEDS_TO_RUN = [
    42,
    # 123,   # Belum dijalankan untuk scratch
    # 2024,  # Belum dijalankan untuk scratch
    # 7,     # Belum dijalankan untuk scratch
    # 3407,  # Belum dijalankan untuk scratch
]
SEED = SEEDS_TO_RUN[0] if SEEDS_TO_RUN else None  # Diatur ulang otomatis per run.
DEVICE = 0                   # GPU Kaggle; 'cpu' untuk diagnosis lokal
WORKERS = 2
OPTIMIZER = "AdamW"
LR = 0.001
PATIENCE = 100
AMP = True

WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 3.0
CLOSE_MOSAIC = 10
EVAL_SPLIT = "val"           # test hanya jika dataset punya split test
PROJECT = "/kaggle/working/results"  # Colab: ganti ke /content/results
TEACHER_CHECKPOINT = None

# W&B online: setiap epoch mengunggah last.pt + best.pt sampai epoch tersebut.
WANDB_ENABLED = True
WANDB_PROJECT = "yolom26-kd-indotraffic"
WANDB_ENTITY = None          # gunakan akun default; ganti nama tim bila perlu

# Group membandingkan pasangan yang sama; tag mengikuti konfigurasi run baru.
WANDB_GROUP = "supervised"
WANDB_TAGS = [
    'baseline', 'model-n', 'from-scratch', f"seed{SEED}", "poltrada-v3", EXPERIMENT_ID
]

BASE_WANDB_TAGS = [tag for tag in WANDB_TAGS if not tag.startswith("seed")]


## 2. Environment Setup

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys
import os

# Pertahankan instalasi PyTorch/CUDA bawaan Kaggle; instal hanya yang dibutuhkan.
for package, version in (("ultralytics", "8.4.155"), ("wandb", "0.28.1")):
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        print(f"Installing {package}=={version} (was {installed})")
        subprocess.check_call([sys.executable, "-m", "pip", "install", f"{package}=={version}"])

import json
import gc
from contextlib import contextmanager
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
import wandb
from ultralytics import YOLO, settings
from ultralytics.utils.torch_utils import init_seeds

# Satu sumber logging W&B: callback eksplisit di section 6, bukan integrasi otomatis Ultralytics.
settings.update({"wandb": False})
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | Ultralytics {metadata.version('ultralytics')} | W&B {metadata.version('wandb')}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")
if DEVICE != "cpu" and not torch.cuda.is_available():
    raise RuntimeError("GPU tidak tersedia. Aktifkan GPU Kaggle/Colab atau set DEVICE='cpu' untuk test lokal.")

if WANDB_ENABLED:
    # Kaggle: tambahkan secret WANDB_API_KEY melalui Add-ons > Secrets dan aktifkan notebook access.
    if not os.environ.get("WANDB_API_KEY"):
        try:
            from kaggle_secrets import UserSecretsClient
            secret = UserSecretsClient().get_secret("WANDB_API_KEY")
            if secret:
                os.environ["WANDB_API_KEY"] = secret
        except ImportError:
            pass  # Colab atau lokal: sediakan WANDB_API_KEY sebagai environment variable.
    if not os.environ.get("WANDB_API_KEY"):
        raise RuntimeError("W&B online membutuhkan WANDB_API_KEY. Set sebagai Kaggle Secret atau environment variable; jangan tulis token ke notebook.")
    wandb.login(key=os.environ["WANDB_API_KEY"])

ENVIRONMENT_INFO = {"python": sys.version, "torch": torch.__version__, "cuda": torch.version.cuda,
                    "ultralytics": metadata.version("ultralytics"), "wandb": metadata.version("wandb")}


## 3. Dataset Loading

In [ ]:
from ultralytics.data.utils import check_det_dataset

# Memakai loader resmi, tanpa audit/cleaning dataset dalam notebook eksperimen.
data_info = check_det_dataset(DATA)
assert int(data_info["nc"]) > 0, "Dataset tidak memiliki kelas deteksi."
assert data_info.get("train") and data_info.get("val"), "Dataset wajib punya train dan val."
if EVAL_SPLIT == "test" and not data_info.get("test"):
    raise ValueError("Dataset ini tidak memiliki test split; gunakan EVAL_SPLIT='val' hanya untuk diagnosis.")

# Periksa kesesuaian nama kelas teacher/dataset untuk metode KD.
def class_names(names):
    if isinstance(names, dict):
        return [str(names[k]) for k in sorted(names, key=lambda k: int(k))]
    return [str(x) for x in names]

print(f"Dataset: {DATA} | classes: {data_info['nc']} | validation: {EVAL_SPLIT}")

## 4. Model Initialization

In [ ]:
def log_status(stage, message):
    stamp = datetime.now().astimezone().strftime("%H:%M:%S")
    print(f"[{stamp}] [{stage}] {message}", flush=True)

print("Student YOLO26n berbobot acak akan dibuat terpisah untuk setiap seed (pretrained=False).")


## 5. Training Method

Tidak ada distillation. Loss deteksi bawaan Ultralytics sama dengan baseline 03.
Satu-satunya perbedaan eksperimen adalah inisialisasi: bobot acak dari arsitektur nano,
bukan bobot pretrained. Optimizer, augmentasi dan jadwal training tidak disesuaikan untuk scratch.


In [ ]:
KD_TRAINER = None
KD_TRAIN_ARGS = {}
KD_CONFIG = {}

## 6. Definisi training dan antrean seed

Cell ini menyiapkan fungsi. Eksekusi serial dimulai di section 8 setelah fungsi evaluasi tersedia.


In [ ]:
@contextmanager
def exclusive_gpu():
    lock = Path(PROJECT).parent / ".yolo26kd_teacher_s_gpu.lock"
    lock.parent.mkdir(parents=True, exist_ok=True)
    try:
        descriptor = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    except FileExistsError as error:
        raise RuntimeError(f"GPU dikunci eksperimen lain: {lock}.") from error
    try:
        with os.fdopen(descriptor, "w") as stream:
            stream.write(json.dumps({"pid": os.getpid(), "experiment": EXPERIMENT_ID, "run": wb_run.id}))
        yield
    finally:
        lock.unlink(missing_ok=True)


def prepare_seed_teacher():
    # Baseline YOLO26n tidak menggunakan teacher.
    pass

def save_epoch_artifact(trainer):
    """Snapshot checkpoint setiap epoch; upload selesai sebelum last.pt ditimpa epoch berikutnya."""
    epoch = int(trainer.epoch) + 1
    last_path = Path(trainer.last)
    best_path = Path(trainer.best)
    if not last_path.is_file():
        raise RuntimeError(f"Checkpoint epoch {epoch} tidak ada: {last_path}")
    fitness = None if trainer.fitness is None else float(trainer.fitness)
    best_fitness = None if trainer.best_fitness is None else float(trainer.best_fitness)
    is_best = fitness is not None and best_fitness is not None and fitness == best_fitness
    if is_best:
        wb_state["best_epoch"] = epoch
    artifact = wandb.Artifact(
        name=f"{EXPERIMENT_ID}-{wb_run.id}-checkpoints",
        type="model",
        description="Checkpoint penuh per epoch: last.pt, best-to-date.pt, results.csv, args.yaml",
        metadata={
            "epoch": epoch, "fitness": fitness, "best_fitness": best_fitness,
            "is_best_epoch": bool(is_best), "best_epoch_so_far": wb_state["best_epoch"],
            "experiment": EXPERIMENT_ID, "dataset": DATA, "seed": SEED,
            "pretrained": PRETRAINED, "initialization": "random",
        },
    )
    artifact.add_file(str(last_path), name="last.pt")
    if best_path.is_file():
        artifact.add_file(str(best_path), name="best.pt")
    for filename in ("results.csv", "args.yaml"):
        file_path = Path(trainer.save_dir) / filename
        if file_path.is_file():
            artifact.add_file(str(file_path), name=filename)
    aliases = [f"epoch-{epoch:03d}", "latest"]
    if is_best:
        aliases.append("best")
    # Synchronous wait menjamin isi artifact epoch ini selesai dikirim sebelum file lokal berubah.
    wb_run.log_artifact(artifact, aliases=aliases).wait()
    wb_state["uploaded_epochs"].add(epoch)
    print(f"W&B checkpoint: epoch={epoch}, best={is_best}, aliases={aliases}")

def log_epoch_metrics(trainer):
    """Satu record grafik per epoch (training loss, KD loss, validation, LR, best epoch)."""
    epoch = int(trainer.epoch) + 1
    # Ultralytics memanggil on_fit_epoch_end sekali lagi pada final_eval; abaikan panggilan ekstra.
    if epoch not in wb_state["uploaded_epochs"] or epoch in wb_state["logged_epochs"]:
        return
    loss = trainer.label_loss_items(trainer.tloss, prefix="train")
    payload = {"epoch": epoch, **{k: float(v) for k, v in loss.items()},
               **{k: float(v) for k, v in trainer.lr.items()}}
    for key, value in trainer.metrics.items():
        try:
            payload[key] = float(value)
        except (ValueError, TypeError):
            continue
    payload["checkpoint/is_best_epoch"] = int(wb_state["best_epoch"] == epoch)
    payload["checkpoint/best_epoch_so_far"] = wb_state["best_epoch"] or 0
    wb_run.log(payload, step=epoch)
    wb_state["logged_epochs"].add(epoch)

def run_full_training():
    model = None
    with exclusive_gpu():
        try:
            init_seeds(SEED, deterministic=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats(DEVICE)
            model = YOLO(MODEL_WEIGHTS)
            train_args = dict(
                data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                optimizer=OPTIMIZER, lr0=LR, seed=SEED, device=DEVICE,
                weight_decay=WEIGHT_DECAY, warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
                project=str(SESSION_DIR), name="training", exist_ok=False,
                workers=WORKERS, patience=PATIENCE, plots=True,
                amp=AMP, pretrained=PRETRAINED, save=True,
            )
            train_args.update(KD_TRAIN_ARGS)
            wb_run.config.update({"training_args": train_args, "kd": KD_CONFIG})
            model.add_callback("on_model_save", bind_seed_callback(save_epoch_artifact))
            model.add_callback("on_fit_epoch_end", bind_seed_callback(log_epoch_metrics))
            start = time.perf_counter()
            model.train(**train_args)
            directory = Path(model.trainer.save_dir)
            best = directory / "weights/best.pt"
            if not best.is_file():
                raise FileNotFoundError(f"Checkpoint best.pt tidak ditemukan: {best}")
            return {"run_dir": str(directory), "best_pt": str(best),
                    "last_pt": str(directory / "weights/last.pt"),
                    "training_seconds": time.perf_counter() - start,
                    "completed_epochs": len(wb_state["uploaded_epochs"]), "best_epoch": wb_state["best_epoch"],
                    "pretrained": PRETRAINED, "initialization": "random"}
        finally:
            model = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


def validate_seed_list(seeds):
    """Reject ambiguous queues before opening any run or creating a model."""
    if not isinstance(seeds, (list, tuple)) or not seeds:
        raise ValueError("SEEDS_TO_RUN harus berupa daftar seed yang tidak kosong.")
    if any(isinstance(seed, bool) or not isinstance(seed, int) or not 0 <= seed < 2**32 for seed in seeds):
        raise ValueError("Setiap seed harus integer dalam rentang 0 sampai 2**32-1.")
    if len(seeds) != len(set(seeds)):
        raise ValueError("SEEDS_TO_RUN tidak boleh memiliki seed duplikat.")
    return list(seeds)


def seed_run_name(seed):
    if METHOD == "none":
        return f"baseline_yolo26n_scratch_seed{seed}"
    if METHOD == "native":
        return f"native_T26s_S26n_dis{KD_WEIGHT:g}_seed{seed}"
    return f"crosskd_T26s_S26n_seed{seed}"


def bind_seed_callback(callback):
    """Bind the active seed/run/state; reject stale callbacks instead of mixing logs."""
    seed, run, state = SEED, wb_run, wb_state
    def bound(trainer):
        if SEED != seed or wb_run is not run or wb_state is not state:
            raise RuntimeError("Callback berasal dari seed/run lain; logging dihentikan.")
        return callback(trainer)
    return bound


def start_seed_run():
    global wb_run
    NEW_WANDB_RUN_ID = wandb.util.generate_id()
    log_status("W&B", f"Membuat run baru: name={WANDB_RUN_NAME}, id={NEW_WANDB_RUN_ID}, resume=never...")
    diagnostic = DATA == "coco8.yaml"
    wb_run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=WANDB_RUN_NAME,
        id=NEW_WANDB_RUN_ID, resume="never", settings=wandb.Settings(init_timeout=90),
        group=f"diagnostic-{WANDB_GROUP}" if diagnostic else WANDB_GROUP, job_type="training",
        tags=[*WANDB_TAGS, "diagnostic" if diagnostic else "research"],
        config={"experiment": EXPERIMENT_ID, "method": METHOD, "model": MODEL_WEIGHTS,
                "pretrained": PRETRAINED, "initialization": "random",
                "dataset": DATA, "eval_split": EVAL_SPLIT, "seed": SEED,
                "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "device": str(DEVICE),
                "workers": WORKERS, "optimizer": OPTIMIZER, "learning_rate": LR,
                "patience": PATIENCE, "amp": AMP, "weight_decay": WEIGHT_DECAY,
                "warmup_epochs": WARMUP_EPOCHS, "close_mosaic": CLOSE_MOSAIC,
                "teacher_artifact": TEACHER_ARTIFACT if METHOD != "none" else None,
                "environment": ENVIRONMENT_INFO, "seeds_to_run": list(SEEDS_TO_RUN)})
    log_status("W&B", f"Run aktif: {wb_run.url}")


def run_one_seed(seed, queue_dir):
    """Complete one independent run, including evaluation/upload/finish, before returning."""
    global SEED, SESSION_DIR, WANDB_RUN_NAME, WANDB_TAGS, wb_run, wb_state
    global TEACHER_CHECKPOINT, TEACHER_PROVENANCE, teacher_path
    if wandb.run is not None:
        raise RuntimeError("Masih ada run W&B aktif; seed baru tidak boleh dimulai.")
    SEED = seed
    WANDB_RUN_NAME = seed_run_name(seed)
    WANDB_TAGS = [*BASE_WANDB_TAGS, f"seed{seed}"]
    SESSION_DIR = Path(queue_dir) / f"seed-{seed}"
    SESSION_DIR.mkdir(parents=True, exist_ok=False)
    wb_run = None
    wb_state = {"saved_epochs": set(), "uploaded_epochs": set(), "logged_epochs": set(),
                "best_epoch": None, "epoch_upload_seconds": []}
    if METHOD != "none":
        TEACHER_CHECKPOINT, TEACHER_PROVENANCE, teacher_path = None, {}, None
        KD_TRAIN_ARGS["distill_model"] = None
    try:
        init_seeds(seed, deterministic=True)
        start_seed_run()
        prepare_seed_teacher()
        log_status("SEED", f"Memulai seed {seed}: {EPOCHS} epoch, student berbobot acak dari {MODEL_WEIGHTS}, pretrained={PRETRAINED}.")
        training = run_full_training()
        (SESSION_DIR / "training_result.json").write_text(json.dumps(training, indent=2), encoding="utf-8")
        with exclusive_gpu():
            result = evaluate_final_training(training)
        result.update(seed=seed, wandb_run_id=wb_run.id, wandb_run_name=WANDB_RUN_NAME,
                      wandb_url=wb_run.url, seed_directory=str(SESSION_DIR))
        (SESSION_DIR / "seed_result.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
        # finish() waits for the run's remaining W&B sync; the next seed starts afterwards.
        wb_run.finish()
        log_status("SEED", f"Seed {seed} selesai; run W&B sudah ditutup.")
        return result
    except BaseException:
        if wb_run is not None:
            try:
                wb_run.finish(exit_code=1)
            except Exception as finish_error:
                log_status("W&B", f"Penutupan run gagal ({type(finish_error).__name__}); antrean dihentikan.")
        raise
    finally:
        wb_run = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def run_seed_queue():
    """Serial queue: persist completed seeds and stop immediately on any failure."""
    seeds = validate_seed_list(SEEDS_TO_RUN)
    if not WANDB_ENABLED:
        raise ValueError("Antrean penelitian ini memerlukan W&B online.")
    if wandb.run is not None:
        raise RuntimeError("Selesaikan run W&B aktif sebelum menjalankan antrean ini.")
    token = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    queue_dir = Path(PROJECT) / EXPERIMENT_ID / f"multiseed_{token}"
    queue_dir.mkdir(parents=True, exist_ok=False)
    results = []
    progress = {"experiment": EXPERIMENT_ID, "seeds_to_run": seeds, "completed": [],
                "failed_seed": None, "status": "running"}
    progress_path = queue_dir / "multiseed_progress.json"
    progress_path.write_text(json.dumps(progress, indent=2), encoding="utf-8")
    log_status("QUEUE", f"Seed serial {seeds}; hasil: {queue_dir}. Eksperimen scratch; hasil pretrained tidak diimpor.")
    for seed in seeds:
        try:
            result = run_one_seed(seed, queue_dir)
            results.append(result)
            (queue_dir / "multiseed_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
            progress["completed"].append({"seed": seed, "wandb_run_id": result["wandb_run_id"],
                                          "seed_directory": result["seed_directory"]})
            progress_path.write_text(json.dumps(progress, indent=2), encoding="utf-8")
        except BaseException:
            progress.update(failed_seed=seed, status="failed")
            progress_path.write_text(json.dumps(progress, indent=2), encoding="utf-8")
            log_status("QUEUE", f"Berhenti pada seed {seed}. Hasil seed selesai tetap tersimpan; untuk lanjut isi SEEDS_TO_RUN dengan seed yang belum selesai.")
            raise
    progress["status"] = "completed"
    progress_path.write_text(json.dumps(progress, indent=2), encoding="utf-8")
    log_status("QUEUE", f"Selesai: {len(results)} run terpisah. Bandingkan dengan baseline pretrained pada seed yang sama; jangan gabungkan dua kondisi inisialisasi.")
    return results


## 7. Model Evaluation

In [ ]:
def evaluate_final_training(training):
    best_model = metrics = None
    try:
        run_dir = Path(training["run_dir"])
        best_pt = Path(training["best_pt"])
        training_seconds = training["training_seconds"]
        # Evaluasi dari checkpoint terbaik (bukan last.pt).
        best_model = YOLO(str(best_pt))
        metrics = best_model.val(
            data=DATA, split=EVAL_SPLIT, imgsz=IMGSZ, batch=BATCH,
            device=DEVICE, plots=True, verbose=False,
        )

        from ultralytics.utils.torch_utils import get_flops
        parameters = sum(p.numel() for p in best_model.model.parameters())
        try:
            gflops = float(get_flops(best_model.model, imgsz=IMGSZ))
        except (AttributeError, ImportError, RuntimeError, TypeError, ValueError) as error:
            print("GFLOPs tidak tersedia:", error)
            gflops = None

        results = {
            "experiment": EXPERIMENT_ID,
            "method": METHOD,
            "model": MODEL_WEIGHTS,
            "pretrained": PRETRAINED, "initialization": "random",
            "dataset": DATA,
            "eval_split": EVAL_SPLIT,
            "seed": SEED,
            "epochs": EPOCHS,
            "imgsz": IMGSZ,
            "batch": BATCH,
            "optimizer": OPTIMIZER,
            "learning_rate": LR,
            "precision": float(metrics.box.mp),
            "recall": float(metrics.box.mr),
            "map50": float(metrics.box.map50),
            "map50_95": float(metrics.box.map),
            "parameters": int(parameters),
            "gflops": gflops,
            "inference_ms_per_image": float(metrics.speed.get("inference", 0.0)),
            "training_seconds": float(training_seconds),
            "device": str(DEVICE),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "teacher_checkpoint": str(TEACHER_CHECKPOINT) if METHOD != "none" else None,
            "best_pt": str(best_pt),
        }
        if torch.cuda.is_available():
            results["peak_gpu_memory_gb"] = float(torch.cuda.max_memory_allocated() / 1024**3)
        print(json.dumps(results, indent=2, ensure_ascii=False))
        # File lokal tetap tersimpan oleh Ultralytics, W&B menyimpan versi artifact tiap epoch.
        run_dir.mkdir(parents=True, exist_ok=True)
        results["best_epoch"] = wb_state["best_epoch"] if WANDB_ENABLED else None
        (run_dir / "metrics.json").write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
        configuration = {
            "experiment": EXPERIMENT_ID, "dataset": DATA, "evaluation_split": EVAL_SPLIT,
            "model": MODEL_WEIGHTS, "teacher": TEACHER_CHECKPOINT if METHOD != "none" else None,
            "pretrained": PRETRAINED, "initialization": "random",
            "seed": SEED, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH,
            "optimizer": OPTIMIZER, "lr0": LR, "device": DEVICE,
            "workers": WORKERS, "patience": PATIENCE, "amp": AMP,
            "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS,
            "close_mosaic": CLOSE_MOSAIC,
            "kd_hyperparameters": KD_CONFIG,
            "wandb_enabled": WANDB_ENABLED, "wandb_project": WANDB_PROJECT,
        }
        (run_dir / "config.json").write_text(json.dumps(configuration, indent=2, ensure_ascii=False), encoding="utf-8")

        if wb_run is not None:
            summary = {f"final/{key}": val for key, val in results.items()
                       if isinstance(val, (int, float)) and not isinstance(val, bool) and val is not None}
            wb_run.log(summary)
            wb_run.summary.update(summary)
            wb_run.summary["best_epoch"] = wb_state["best_epoch"]
            wb_run.summary["checkpoint_collection"] = f"{EXPERIMENT_ID}-{wb_run.id}-checkpoints"
            wb_run.summary["uploaded_checkpoint_epochs"] = len(wb_state["uploaded_epochs"])

            # Ringkasan/config dan grafik sebagai Artifact terpisah, tanpa menggandakan checkpoint model.
            final_artifact = wandb.Artifact(
                name=f"{EXPERIMENT_ID}-{wb_run.id}-results", type="results",
                metadata={"experiment": EXPERIMENT_ID, "seed": SEED, "best_epoch": wb_state["best_epoch"],
                          "pretrained": PRETRAINED, "initialization": "random"},
            )
            for filename in ("metrics.json", "config.json", "results.csv", "args.yaml"):
                candidate = run_dir / filename
                if candidate.is_file():
                    final_artifact.add_file(str(candidate), name=filename)
            for folder_name, folder in (("training", run_dir),
                                        ("evaluation", Path(getattr(metrics, "save_dir", run_dir)))):
                if folder.exists():
                    for plot_path in sorted(folder.glob("*.png")):
                        key = f"plots/{folder_name}/{plot_path.stem}"
                        wb_run.log({key: wandb.Image(str(plot_path))})
                        final_artifact.add_file(str(plot_path), name=f"{folder_name}/{plot_path.name}")
            wb_run.log_artifact(final_artifact, aliases=["final"]).wait()
            print("W&B:", wb_run.url)
            print("Checkpoint alias: epoch-001 ... epoch-%03d, latest, best" % len(wb_state["uploaded_epochs"]))

        print("Hasil lokal:", run_dir)
        print("  - weights/best.pt, weights/last.pt")
        print("  - results.csv, args.yaml")
        print("  - metrics.json, config.json")

        return results

    finally:
        best_model = metrics = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


## 8. Jalankan training + evaluasi serial dan simpan hasil

Default antrean menjalankan seed 42 saja untuk scratch. Bandingkan dengan run pretrained seed 42
pada dataset, split, jumlah epoch dan protokol yang sama. Seed lain belum dianggap selesai;
aktifkan komentar pada konfigurasi untuk memperluas pengujian secara serial.

Output berada di direktori eksperimen `03_baseline_yolo26n_scratch`, dengan run W&B
`baseline_yolo26n_scratch_seed{seed}`, group `supervised`, tag `from-scratch` dan ID run baru.
Hasil disimpan terpisah; tidak ada penggabungan otomatis dengan hasil pretrained.
Jika antrean gagal, jalankan kembali hanya seed scratch yang belum selesai.
Setelah beberapa seed tersedia, hitung mean ± standar deviasi secara terpisah untuk tiap kondisi.


In [ ]:
MULTISEED_RESULTS = run_seed_queue()
print(json.dumps(MULTISEED_RESULTS, indent=2))
